# HW1 Submission 

**Before you start, copy this notebook into your own workspace folder.** 

Follow this template for Questions 1, 2, 3 and the evidence section of Question 4, filling in your code in the cells indicated. Feel free to add as many cells as you want, but keep the same layout and do not remove the section headings.

For Question 2, you may draft each query in the `%sql` cell provided, then run it with `save_query`, which writes the `.sql` and `.csv` files you submit.
The written parts of Question 4 are submitted as a PDF; only the code that produced your numbers goes here.

Split your work into several cells rather than writing one long one, and add a short comment where a step is not obvious from the code itself. Do not write an essay in comments: a line explaining why you filtered something, or what a parsing step returns, is enough.

Everything you submit must be what this notebook produced. The files are written by the helpers provided below, not by hand, and the numbers in your Question 4 report must come from code in the evidence section. Do not edit a saved file afterwards, and do not report a number the notebook cannot reproduce. We check this.

Before you submit: restart the notebook and run it from beginning to end. It must complete without errors, and every output must be visible in the exported HTML.

## Load the data

Run the two cells below to load the data.

### Load the DataFrame

Run the cell below to read the data into `airbnb_df`. Use this for Questions 1, 3 and 4.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

DATA_PATH = "/Volumes/94290_lab/course_data/airbnb/airbnb_1_12_parquet"
airbnb_df = spark.read.parquet(DATA_PATH)
print(f"rows: {airbnb_df.count():,}   columns: {len(airbnb_df.columns)}")
display(airbnb_df.limit(5))

%md
### Load the SQL view

Run the cell below to make the same data available to the `%sql` cells as `airbnb_data`. Use this for Question 2.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW airbnb_data AS
SELECT * FROM parquet.`/Volumes/94290_lab/course_data/airbnb/airbnb_1_12_parquet`;
SELECT * FROM airbnb_data LIMIT 5


## Question 1: Data Handling

PySpark only. No Pandas, NumPy or similar libraries.

Each question produces one table, saved with the `save_csv` helper below. Run the next cell once before you start.

In [ ]:
import os

try:
    # inside a Databricks notebook this is the folder the notebook lives in
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext()\
        .notebookPath().get()
    OUT_DIR = "/Workspace" + notebook_path.rsplit("/", 1)[0]
except Exception:
    # Databricks Connect has no notebook path; dbutils raises "cluster_id is required"
    OUT_DIR = os.getcwd()
print(OUT_DIR)


def wait_for_file(path, timeout=30):
    """Wait until a written file is visible and non-empty. Do not modify."""
    import os, time
    start = time.time()
    while time.time() - start < timeout:
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
        time.sleep(0.5)
    raise TimeoutError(f"{path} did not appear within {timeout}s")


def save_csv(sdf, file_name, out_dir=OUT_DIR):
    """Write a small result table to <out_dir>/<file_name>.csv. Do not modify this function."""
    import os, csv
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{file_name}.csv")
    rows = sdf.collect()
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
        w.writerow(sdf.columns)
        for r in rows:
            w.writerow(["" if v is None else v for v in r])
        f.flush()
        os.fsync(f.fileno())
    wait_for_file(path)
    print(f"wrote {len(rows)} rows to {path}")
    return path


def save_query(query, file_name, out_dir=OUT_DIR):
    """Run a SQL query, save the query text as .sql and its result as .csv. Do not modify."""
    import os
    os.makedirs(out_dir, exist_ok=True)
    q = query.strip().rstrip(";")
    sql_path = os.path.join(out_dir, f"{file_name}.sql")
    with open(sql_path, "w", encoding="utf-8") as f:
        f.write(q + "\n")
        f.flush()
        os.fsync(f.fileno())
    wait_for_file(sql_path)
    result = spark.sql(q)
    save_csv(result, file_name)
    return result


def show_saved(file_name, out_dir=OUT_DIR):
    """Read back a saved CSV and display it. Do not modify."""
    import csv
    with open(f"{out_dir}/{file_name}.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    print(f"{len(rows) - 1} data rows")
    return spark.createDataFrame(rows[1:], rows[0])

%md
### Question 1a

The 15 properties with the largest gap between their highest and lowest category rating.

Your final DataFrame must be called `q1a`, with the columns already in the required order.

In [ ]:
cat_schema = ArrayType(StructType([
    StructField("name", StringType()),
    StructField("value", StringType()),
]))

q1a_base = (
    airbnb_df
    .withColumn("overall_rating", F.col("ratings").cast("double")) # col 3
    .withColumn("cats_raw", F.from_json("category_rating", cat_schema))
    .withColumn("cats", F.transform("cats_raw", lambda c: F.struct(
        c["name"].alias("name"),
        F.regexp_replace(c["value"], ",", ".").try_cast("double").alias("value"), # def 1
    )))
)

q1a_base.select("property_id", "overall_rating", "cats").printSchema()

In [ ]:
q1a_work = ( ### col 1, 2?
    q1a_base
    .withColumn("cats_present", F.filter("cats", lambda c: c["name"].isNotNull() & c["value"].isNotNull())) # def 1 ### numeric rating?
    .where((F.col("overall_rating") >= 4.0) & (F.col("overall_rating") < 5.0)) # filter 1
    .where(F.exists("cats_present", lambda c: c["name"] == "Cleanliness")) # filter 2
    .where(F.col("property_id").isNotNull()) # filter 3
    .withColumn("num_categories", F.size("cats_present")) # col 4
    .withColumn("best_category_value", F.array_max(F.transform("cats_present", lambda c: c["value"]))) # col 6
    .withColumn("worst_category_value", F.array_min(F.transform("cats_present", lambda c: c["value"]))) # col 8
    .withColumn("best_category", F.array_min(F.transform(
        F.filter("cats_present", lambda c: c["value"] == F.col("best_category_value")),
        lambda c: c["name"]))) # col 5, def 3
    .withColumn("worst_category", F.array_min(F.transform(
        F.filter("cats_present", lambda c: c["value"] == F.col("worst_category_value")),
        lambda c: c["name"]))) # col 7, def 3
    .withColumn("rating_gap", F.col("best_category_value") - F.col("worst_category_value")) # col 9, def 2
)

In [ ]:
q1a = (
    q1a_work
    .select(
        F.col("property_id").cast("string"),
        F.col("listing_name").cast("string"),
        F.col("overall_rating").cast("double"),
        F.col("num_categories").cast("int"),
        F.col("best_category").cast("string"),
        F.col("best_category_value").cast("double"),
        F.col("worst_category").cast("string"),
        F.col("worst_category_value").cast("double"),
        F.col("rating_gap").cast("double"),
    )
    .orderBy(
        F.desc("rating_gap"),
        F.desc("overall_rating"),
        F.asc("property_id"),
        F.desc("num_categories"),
        F.asc("best_category"),
    )
    .limit(15)
)

q1a.printSchema()
display(q1a)

In [0]:
save_csv(q1a, "q1a")
display(show_saved("q1a"))

### Question 1b

The 25 properties with the most beds that do not offer TV and allow check-in at 2:00 PM, among properties priced from 100 to 400 US dollars per night, both ends included, with 4 or more beds.

Your final DataFrame must be called `q1b`, with the columns already in the required order.

In [0]:
q1b_price = airbnb_df.withColumn(
    "price_usd",
    F.when(F.col("currency") == "USD", F.col("price").cast("double"))
     .when(F.col("currency") == "EUR", F.col("price").cast("double") * 1.08)
) # def 4, col 3

In [ ]:
amen_item = StructType([StructField("name", StringType()), StructField("value", StringType())])
amen_schema = ArrayType(StructType([
    StructField("group_name", StringType()),
    StructField("items", ArrayType(amen_item)),
])) # def 2

NOT_INCLUDED_GROUPS = ["Not included", "Nicht enthalten", "Non inclus"]
MISSING_VALUE_RE = (
    r"^SYSTEM_NO_|^SYSTEM_OFFLINE$"
    r"|^There is no .* on the (property|bedroom door)\.$|^This place may not have a |^Host has indicated (that )?no "
    r"|^Es gibt kein|gibt es möglicherweise kein|hat angegeben, dass kein"
) # def 1


def is_available(group, item):
    missing = (F.coalesce(group["group_name"].isin(NOT_INCLUDED_GROUPS), F.lit(False))
               | F.coalesce(item["value"].rlike(MISSING_VALUE_RE), F.lit(False)))
    return ~missing


q1b_amen = (
    q1b_price
    .withColumn("amen_groups", F.from_json("amenities", amen_schema))
    # flatten groups into one list of items per row, tagging each item as available or not
    .withColumn("amen_items", F.flatten(F.transform("amen_groups", lambda g: F.transform(
        g["items"], lambda i: F.struct(
            i["name"].alias("name"),
            i["value"].alias("value"),
            is_available(g, i).alias("available"),
        )))))
    .withColumn("amen_available", F.filter("amen_items", lambda i: i["available"])) # def 1
    .withColumn("num_amenities", F.coalesce(F.size("amen_available"), F.lit(0))) # def 2, col 5
    .withColumn("has_tv", F.coalesce(F.exists("amen_available", lambda i: i["value"] == "SYSTEM_TV"), F.lit(False))) # filter 2
)

In [ ]:
BEDS_RE = r"^(\d+) ([A-Za-z ]+ )?beds?$"

q1b_beds = (# def 3
    q1b_amen
    .withColumn("details_arr", F.from_json("details", ArrayType(StringType())))
    .withColumn("beds_entry", F.get(F.filter("details_arr", lambda d: d.rlike(BEDS_RE)), 0))
    .withColumn("num_beds", F.regexp_extract("beds_entry", BEDS_RE, 1).try_cast("int")) # col 4 
)

In [ ]:
# def 5
def normalise_rule(c): 
    """For parsing only: narrow/no-break spaces -> space, lower case, "p.m." -> "pm"."""
    c = F.regexp_replace(c, "[\u202f\u00a0]", " ")
    return F.regexp_replace(F.lower(c), r"([ap])\.m\.", "$1m")


TIME = r"(\d{1,2}):(\d{2}) ?(am|pm)?"
AFTER_RE = rf"^check-in after {TIME}$"
RANGE_RE = rf"^check-in: {TIME} ?[-–] ?{TIME}$"
FLEX_RE = r"^flexible check-in$"
CHECKIN_RE = f"{AFTER_RE}|{RANGE_RE}|{FLEX_RE}"


def to_minutes(rule, regex, first_group):
    """Minutes after midnight from the hour, minute and am/pm groups (24-hour clock when am/pm is missing)."""
    hour = F.regexp_extract(rule, regex, first_group).try_cast("int")
    minute = F.regexp_extract(rule, regex, first_group + 1).try_cast("int")
    ampm = F.regexp_extract(rule, regex, first_group + 2)
    hour = F.when(ampm == "pm", hour % 12 + 12).when(ampm == "am", hour % 12).otherwise(hour)
    return hour * 60 + minute


TWO_PM = 14 * 60

q1b_checkin = (
    q1b_beds
    .withColumn("rules_arr", F.from_json("house_rules", ArrayType(StringType())))
    .withColumn("check_in_rule", F.get(F.filter("rules_arr", lambda r: normalise_rule(r).rlike(CHECKIN_RE)), 0)) # col 6
    .withColumn("ci_norm", normalise_rule(F.col("check_in_rule")))
    .withColumn("ci_start", F.coalesce(to_minutes(F.col("ci_norm"), AFTER_RE, 1),
                                       to_minutes(F.col("ci_norm"), RANGE_RE, 1)))
    .withColumn("ci_end", to_minutes(F.col("ci_norm"), RANGE_RE, 4))
    .withColumn(
        "allows_2pm",
        F.when(F.col("ci_norm").rlike(FLEX_RE), True)
         .when(F.col("ci_end").isNull(), F.col("ci_start") <= TWO_PM)
         .when(F.col("ci_end") > F.col("ci_start"),
               (F.col("ci_start") <= TWO_PM) & (F.col("ci_end") >= TWO_PM))
         .otherwise((F.col("ci_start") <= TWO_PM) | (F.col("ci_end") >= TWO_PM))
    )
)

In [ ]:
# null price_usd / num_beds / allows_2pm fail these comparisons, so those rows are excluded
q1b = (
    q1b_checkin
    .where(F.col("price_usd").between(100, 400)) # filter 1
    .where(~F.col("has_tv")) # filter 2
    .where(F.col("allows_2pm")) # filter 3 
    .where(F.col("num_beds") >= 4) # filter 4
    .select(
        F.col("property_id").cast("string"), 
        F.col("listing_name").cast("string"),
        F.col("price_usd").cast("double"),
        F.col("num_beds").cast("int"),
        F.col("num_amenities").cast("int"),
        F.col("check_in_rule").cast("string"),
    )
    .orderBy(
        F.desc("num_beds"),
        F.asc("price_usd"),
        F.desc("num_amenities"),
        F.asc("property_id"),
        F.asc("listing_name"),
        F.asc("check_in_rule"),
    )
    .limit(25)
)

q1b.printSchema()
display(q1b)

In [0]:
save_csv(q1b, "q1b")

# read the saved file back and show it, so what you see is what will be submitted
display(show_saved("q1b"))

## Question 2: SQL Queries

Write each query as a string in the cell provided, then run it with `save_query`, which runs the query, saves the text as `q2a.sql` or `q2b.sql`, and saves the result as `q2a.csv` or `q2b.csv`.

Each query must be a single statement that runs on its own and returns exactly the columns listed, with those names, in that order. Use `WITH` clauses for intermediate steps.

You can develop your query in a `%sql` cell first if you prefer the editor, then paste the finished query into the string below. If `airbnb_data` is not found, re-run the setup cell at the top of the notebook.

### Question 2a

The 10 hosts with the highest average rating across their great value listings.


In [0]:
%sql
-- YOUR QUERY HERE

In [0]:
q2a_query = """
 YOUR QUERY HERE 
"""

save_query(q2a_query, "q2a")

# read the saved file back and show it, so what you see is what will be submitted
display(show_saved("q2a"))

### Question 2b

The 10 superhosts whose listings are the most expensive on average.



In [0]:
%sql
-- YOUR QUERY HERE

In [0]:
q2b_query = """
 YOUR QUERY HERE
"""

save_query(q2b_query, "q2b")

# read the saved file back and show it, so what you see is what will be submitted
display(show_saved("q2b"))

## Question 3: Data Exploration & Visualization

Exploration and filtering in PySpark only. Charts may use matplotlib, seaborn, plotly or a dashboarding tool; if you use an external tool, the code that produced the aggregated data must still appear here.

At most 2 charts per section. A figure made of several related subplots counts as one chart, as long as the subplots answer the same question together, for example the same comparison across a few categories, or one distribution split by group. Say in your explanation why the subplots belong together. Unrelated subplots packed into one figure count separately.

For each chart, state the filtering or sampling you applied and how many rows were left, and keep your explanation to at most 3 sentences.

### Question 3a

How do the textual reviews of highly rated properties differ from those of poorly rated ones?

In [0]:
# YOUR CODE HERE

**Assumptions and filtering (3a):**

- What counts as highly rated / poorly rated:
- Filtering or sampling applied, and rows remaining:
- Size of each group:

**What the chart shows:**

### Question 3b

How does the set of amenities offered differ between two groups of listings types that you choose to compare?

In [0]:
# YOUR CODE HERE

**Assumptions and filtering (3b):**

- The two groups being compared, and why they are worth comparing:
- Filtering or sampling applied, and rows remaining:
- Size of each group:

**What the chart shows:**

### Question 3c

Your own question, answered with a visualization.

**Our question:**

**Why it is worth asking:**

In [0]:
# 3c setup: city / country / region keys, price per guest in USD, rating
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import Window
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

# `location` is "City, Region, Country"; ~1% of rows write the country in German (or French).
COUNTRY_FIX = {
    "Vereinigte Staaten": "United States", "États-Unis": "United States", "Frankreich": "France",
    "Italien": "Italy", "Brasilien": "Brazil", "Spanien": "Spain", "Vereinigtes Königreich": "United Kingdom",
    "Mexiko": "Mexico", "Deutschland": "Germany", "Kanada": "Canada", "Griechenland": "Greece",
    "Australien": "Australia", "Kroatien": "Croatia", "Kolumbien": "Colombia", "Philippinen": "Philippines",
    "Indien": "India", "Südkorea": "South Korea", "Türkei": "Turkey", "Argentinien": "Argentina",
    "Polen": "Poland", "Südafrika": "South Africa", "Norwegen": "Norway", "Schweden": "Sweden",
    "Dänemark": "Denmark", "Niederlande": "Netherlands", "Neuseeland": "New Zealand", "Marokko": "Morocco",
    "Schweiz": "Switzerland", "Österreich": "Austria", "Dominikanische Republik": "Dominican Republic",
    "Belgien": "Belgium", "Irland": "Ireland", "Rumänien": "Romania", "Ungarn": "Hungary",
    "Georgien": "Georgia", "Finnland": "Finland", "Vereinigte Arabische Emirate": "United Arab Emirates",
    "Tschechien": "Czechia", "Ägypten": "Egypt", "Albanien": "Albania", "Indonesien": "Indonesia",
    "Slowenien": "Slovenia", "Bulgarien": "Bulgaria", "Kenia": "Kenya", "Serbien": "Serbia", "Kuba": "Cuba",
    "Island": "Iceland", "Jamaika": "Jamaica", "Zypern": "Cyprus", "Saudi-Arabien": "Saudi Arabia",
}

# Regions follow the UN M49 grouping (so Turkey, Cyprus and the Caucasus are in Asia).
# Countries not listed (Canada, Oceania, Africa, small territories) fall into "Other".
REGIONS = {
    "USA": ["United States"],
    "Europe": [
        "France", "Italy", "Spain", "United Kingdom", "Greece", "Germany", "Portugal", "Croatia", "Poland",
        "Norway", "Sweden", "Denmark", "Netherlands", "Switzerland", "Austria", "Belgium", "Ireland", "Czechia",
        "Romania", "Finland", "Hungary", "Russia", "Albania", "Montenegro", "Bulgaria", "Ukraine", "Serbia",
        "Malta", "Slovenia", "Iceland", "Slovakia", "Bosnia & Herzegovina", "Lithuania", "Estonia", "Latvia",
        "North Macedonia", "Moldova", "Luxembourg", "Kosovo", "Faroe Islands", "Andorra", "Belarus",
        "Isle of Man", "Jersey", "Guernsey", "Gibraltar", "Monaco",
    ],
    "Asia": [
        "India", "Thailand", "South Korea", "Philippines", "Malaysia", "Turkey", "Japan", "Vietnam",
        "United Arab Emirates", "Israel", "Georgia", "Saudi Arabia", "Taiwan", "Sri Lanka", "Cyprus", "Lebanon",
        "Pakistan", "Kazakhstan", "Armenia", "Hong Kong", "Jordan", "Cambodia", "Azerbaijan", "Nepal",
        "Uzbekistan", "Oman", "Singapore", "Kyrgyzstan", "Maldives", "Bangladesh", "Bahrain", "Mongolia",
        "Laos", "Qatar", "Indonesia", "Iraq", "Kuwait", "Tajikistan", "Brunei",
    ],
    "Latin America": [
        "Brazil", "Mexico", "Colombia", "Argentina", "Chile", "Costa Rica", "Peru", "Dominican Republic",
        "Ecuador", "Puerto Rico", "Uruguay", "Guatemala", "Cuba", "Jamaica", "Panama", "El Salvador", "Bolivia",
        "Honduras", "Venezuela", "Paraguay", "Nicaragua", "Belize",
    ],
}
REGION_ORDER = ["USA", "Europe", "Asia", "Latin America", "Other"]
PLOT_REGIONS = REGION_ORDER[:4]  # "Other" mixes Canada, Africa and Oceania (23 cities): too mixed to read as one group

# The same city is split under several names; merge the two big cases.
NYC_BOROUGHS = ["Brooklyn", "Manhattan", "Queens", "Bronx", "The Bronx", "Staten Island"]

fix_map = F.create_map(*[F.lit(x) for kv in COUNTRY_FIX.items() for x in kv])
region_map = F.create_map(*[F.lit(x) for r, cs in REGIONS.items() for c in cs for x in (c, r)])
loc_parts = F.split(F.col("location"), ", ")

c3 = (
    airbnb_df
    .where(F.size(loc_parts) >= 2)  # need at least "city, country"
    .withColumn("country_raw", F.trim(F.element_at(loc_parts, -1)))
    .withColumn("country", F.coalesce(F.try_element_at(fix_map, F.col("country_raw")), F.col("country_raw")))
    .withColumn("city", F.trim(loc_parts[0]))
    .withColumn("city", F.when((F.col("country") == "United Kingdom") & (F.col("city") == "Greater London"), "London")
                         .when((F.col("country") == "United States") & F.col("city").isin(NYC_BOROUGHS), "New York")
                         .otherwise(F.col("city")))
    .withColumn("region", F.coalesce(F.try_element_at(region_map, F.col("country")), F.lit("Other")))
    .withColumn("rating", F.col("ratings").try_cast("double"))
    .withColumn("guests_n", F.col("guests").try_cast("int"))
    # `price` is the total of the searched stay when the scrape searched many nights (e.g. 30-night
    # searches in New York), so we use the nightly price from pricing_details instead (see check below)
    .withColumn("nightly", F.get_json_object("pricing_details", "$.price_per_night").try_cast("double"))
    .withColumn("search_nights", F.get_json_object("pricing_details", "$.num_of_nights").try_cast("int"))
    # same USD rule as Q1b: USD unchanged, EUR x 1.08, anything else (null currency) -> null
    .withColumn("price_usd", F.when(F.col("currency") == "USD", F.col("nightly"))
                              .when(F.col("currency") == "EUR", F.col("nightly") * 1.08))
    # price per guest removes most of the "big villas vs studios" difference between cities
    .withColumn("ppg", F.col("price_usd") / F.col("guests_n"))
)

# Why not `price`: how often it is far above the nightly price, and for which search lengths
p_raw = F.col("price").try_cast("double")
inflated = c3.where(p_raw > 1.5 * F.col("nightly"))
print(f"rows where price > 1.5 x price_per_night: {inflated.count():,}")
inflated.groupBy("search_nights").count().orderBy(F.desc("count")).show(5)
c3.where(F.col("city") == "New York").select(
    F.percentile_approx(p_raw, 0.5).alias("median_price_field"),
    F.percentile_approx("nightly", 0.5).alias("median_price_per_night")).show()

In [0]:
# 3c aggregation: one row per city with >= MIN_PROPS distinct properties
MIN_PROPS = 500          # enough listings for a stable median and rating share
PRICE_RANGE = (10, 10000)  # nightly USD; outside this is almost always a scraping artefact (e.g. price=3)
KEYS = ["region", "country", "city"]

# rows are kept as independent records (no de-duplication); competition = distinct properties in the city
priced = c3.where(F.col("price_usd").between(*PRICE_RANGE) & (F.col("guests_n") >= 1))
rated = c3.where(F.col("rating") > 0)  # ratings == 0 is the placeholder for listings with no reviews

country_price = priced.groupBy("country").agg(F.percentile_approx("ppg", 0.5, 10000).alias("country_med_ppg"))

cities_sdf = (
    c3.groupBy(KEYS).agg(F.countDistinct("property_id").alias("n_props"), F.count("*").alias("n_rows"))
    .where(F.col("n_props") >= MIN_PROPS)
    .join(priced.groupBy(KEYS).agg(F.percentile_approx("ppg", 0.5, 10000).alias("city_med_ppg"),
                                   F.count("*").alias("n_priced_rows")), KEYS, "left")
    .join(rated.groupBy(KEYS).agg(F.avg((F.col("rating") >= 4.9).cast("double")).alias("share_top"),
                                  F.avg("rating").alias("mean_rating"),
                                  F.count("*").alias("n_rated_rows")), KEYS, "left")
    .join(country_price, "country", "left")
    .withColumn("rel_price", F.col("city_med_ppg") / F.col("country_med_ppg"))
)

# Spearman correlation per region (rank correlation, robust to the long tail of n_props)
region_rank = lambda c: Window.partitionBy("region").orderBy(c)
spearman = (
    cities_sdf
    .withColumn("r_n", F.rank().over(region_rank("n_props")))
    .withColumn("r_price", F.rank().over(region_rank("rel_price")))
    .withColumn("r_top", F.rank().over(region_rank("share_top")))
    .groupBy("region").agg(F.count("*").alias("n_cities"),
                           F.sum("n_rows").alias("rows_covered"),
                           F.corr("r_n", "r_price").alias("rho_price"),
                           F.corr("r_n", "r_top").alias("rho_top"))
)

print(f"rows with city+country: {c3.count():,} | priced rows: {priced.count():,} | rated rows: {rated.count():,}")
cities = cities_sdf.toPandas()   # already aggregated: one row per city
rho = spearman.toPandas().set_index("region").reindex(REGION_ORDER)
print(f"cities with >= {MIN_PROPS} properties: {len(cities)}, covering {cities.n_rows.sum():,} rows")
# printed as text (not display) so the tables survive the HTML export
print(rho.round(3).to_string())
print(cities.sort_values("n_props", ascending=False).head(15).round(3).to_string(index=False))

In [0]:
# 3c chart 1: competition vs relative price (top row) and vs share of top-rated listings (bottom row)
colors = dict(zip(PLOT_REGIONS, ["#1f77b4", "#2ca02c", "#d62728", "#ff7f0e"]))
fig, axes = plt.subplots(2, 4, figsize=(19, 8.5), sharex=True, sharey="row")
for j, region in enumerate(PLOT_REGIONS):
    d = cities[cities.region == region]
    for i, (col, rho_col) in enumerate([("rel_price", "rho_price"), ("share_top", "rho_top")]):
        ax = axes[i, j]
        ax.scatter(d.n_props, d[col], s=18, alpha=0.6, color=colors[region], edgecolor="none")
        if len(d) > 2:  # least-squares trend on log10(n_props), drawn only as a visual guide
            k, b = np.polyfit(np.log10(d.n_props), d[col], 1)
            xs = np.logspace(np.log10(d.n_props.min()), np.log10(d.n_props.max()), 50)
            ax.plot(xs, k * np.log10(xs) + b, color="black", lw=1.5)
        for _, r in d.nlargest(3, "n_props").iterrows():
            ax.annotate(r.city, (r.n_props, r[col]), fontsize=8, xytext=(3, 3), textcoords="offset points")
        ax.set_xscale("log")
        ax.set_title(f"{region} (n={len(d)} cities, ρ={rho.loc[region, rho_col]:+.2f})", fontsize=11)
        if i == 0:
            ax.axhline(1, color="grey", ls="--", lw=1)
axes[0, 0].set_ylabel("city median price per guest\n÷ country median")
axes[1, 0].set_ylabel("share of rated listings\nwith rating ≥ 4.9")
for ax in axes[1]:
    ax.set_xlabel("distinct properties in city (log)")
axes[0, 0].set_yscale("log")
axes[0, 0].set_yticks([0.5, 0.75, 1, 1.5, 2, 3], ["0.5", "0.75", "1", "1.5", "2", "3"])
axes[0, 0].minorticks_off()
fig.suptitle("Does more Airbnb competition in a city mean cheaper or better-rated stays?", fontsize=14)
fig.tight_layout()
plt.show()

**Assumptions and filtering (3c):**

- Filtering or sampling applied, and rows remaining:

**What the chart shows:**

In [0]:
# 3c follow-up: what else changes with competition? Per-city mix and category ratings, again one row per city
CATS = ["Cleanliness", "Accuracy", "Check-in", "Communication", "Location", "Value"]  # English names only
cat_schema_3c = ArrayType(StructType([StructField("name", StringType()), StructField("value", StringType())]))
cat_val = lambda name: F.get(F.transform(F.filter("cats", lambda c: c["name"] == name),
                                          lambda c: F.regexp_replace(c["value"], ",", ".").try_cast("double")), 0)

seller = F.get_json_object("seller_info", "$.seller_id")
# host size = distinct properties the host has in the dataset (one row per seller, so the join adds no rows)
host_size = (airbnb_df.where(seller.isNotNull()).groupBy(seller.alias("seller_id"))
             .agg(F.countDistinct("property_id").alias("host_props")))

c3x = (
    c3.withColumn("seller_id", seller)
    .join(host_size, "seller_id", "left")
    .withColumn("cats", F.from_json("category_rating", cat_schema_3c))
    .select(*KEYS, "rating", "host_props", "is_supperhost", "listing_name",
            *[cat_val(c).alias(c) for c in CATS])
)
assert c3x.count() == c3.count(), "host join must not add rows"
mix = (
    c3x.groupBy(KEYS).agg(
        *[F.avg(c).alias(c) for c in CATS],  # avg ignores rows without that category
        F.avg((F.col("host_props") >= 10).cast("double")).alias("share_pro_host"),
        F.avg((F.col("is_supperhost") == "true").cast("double")).alias("share_superhost"),
        F.avg(F.col("listing_name").startswith("Entire").cast("double")).alias("share_entire_place"),
    )
    .join(cities_sdf.select(*KEYS, "n_props", "share_top", "rel_price"), KEYS, "inner")  # keep the same cities as chart 1
)

METRICS = {"rel_price": "relative price per guest", "share_top": "share rated ≥ 4.9",
           **{c: f"{c} rating" for c in CATS},
           "share_superhost": "share superhost listings", "share_pro_host": "share from hosts with ≥10 listings",
           "share_entire_place": "share entire places"}
rmix = mix.withColumn("r_n", F.rank().over(region_rank("n_props")))
for m in METRICS:
    rmix = rmix.withColumn(f"r_{m}", F.rank().over(region_rank(m)))
rho_mix = (rmix.where(F.col("region").isin(PLOT_REGIONS)).groupBy("region")
           .agg(F.count("*").alias("n_cities"), *[F.corr("r_n", f"r_{m}").alias(m) for m in METRICS])
           .toPandas().set_index("region").reindex(PLOT_REGIONS))
print(rho_mix.round(3).to_string())

In [0]:
# 3c chart 2: heatmap of Spearman rho between city size and each city-level metric, per region
mat = rho_mix[list(METRICS)].T
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(mat.values, cmap="RdBu", vmin=-0.6, vmax=0.6, aspect="auto")
for i in range(mat.shape[0]):
    for j, region in enumerate(mat.columns):
        v = mat.iloc[i, j]
        sig = abs(v) > 1.96 / np.sqrt(rho_mix.loc[region, "n_cities"] - 1)  # approx. 95% test for rho = 0
        ax.text(j, i, f"{v:+.2f}{'*' if sig else ''}", ha="center", va="center", fontsize=10,
                color="white" if abs(v) > 0.4 else "black")
ax.set_xticks(range(len(mat.columns)), [f"{r}\n({int(rho_mix.loc[r, 'n_cities'])} cities)" for r in mat.columns])
ax.set_yticks(range(len(mat.index)), [METRICS[m] for m in mat.index])
ax.axhline(1.5, color="black", lw=1); ax.axhline(7.5, color="black", lw=1)
fig.colorbar(im, ax=ax, label="Spearman ρ with number of properties in city")
ax.set_title("As a city's Airbnb market grows, what changes?\n(* = |ρ| significant at ~95%)")
fig.tight_layout()
plt.show()

**Assumptions and filtering (3c, chart 2):**

- Filtering or sampling applied, and rows remaining:

**What the chart shows:**

## Question 4 Evidence

The written report goes in `q4.pdf`. This section holds only the code that produced the numbers and examples you cite there. Every number in your report must be reproducible from a cell below.

### 4a, problem 1

**Problem:** (one line, matching the heading in your PDF)

In [0]:
# YOUR CODE HERE

### 4a, problem 2

**Problem:**

In [0]:
# YOUR CODE HERE

### 4a, problem 3

**Problem:**

In [0]:
# YOUR CODE HERE

### 4b: simple supporting evidence for your research question

**Research question:**

In [0]:
# YOUR CODE HERE